### Distributed Model Training

How training scales across multiple devices once a single GPU can't hold the data, the model, or both: data parallelism (DDP), model parallelism, pipeline parallelism, and the sync-vs-async gradient aggregation choice. Distinct from `data-pipelines-and-scaling.ipynb`'s PySpark section, which is about scaling ETL over a dataset, not distributing the training computation itself. See `pytorch-intro.ipynb` for the single-GPU training loop this generalizes.

## Data parallelism / DistributedDataParallel (DDP)

#### 0. Core idea

The model is small enough to fit on one device, but the DATASET or batch is large. Copy the full model onto every worker (GPU), split each batch into shards (one shard per worker), have each worker compute gradients on its own shard independently, then average ("all-reduce") the gradients across all workers before applying the update, so every worker ends the step with the exact same, updated model.

The averaged gradient from N equal-sized shards is mathematically identical to the gradient you'd have gotten from running the whole batch on one device, since averaging shard-level means of equal-sized shards is the same as the overall mean.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = rng.normal(size=8)
y = 2*x + rng.normal(scale=0.1, size=8)
w = 0.5  # current parameter

def mse_grad(xs, ys, w):
    preds = w*xs
    return np.mean(2*(preds - ys)*xs)

full_batch_grad = mse_grad(x, y, w)

# split into 2 shards (2 "workers"), compute each independently
g_worker1 = mse_grad(x[:4], y[:4], w)
g_worker2 = mse_grad(x[4:], y[4:], w)
allreduced_grad = (g_worker1 + g_worker2) / 2

print("full-batch gradient:       ", full_batch_grad)
print("worker 1 / worker 2 grads: ", g_worker1, g_worker2)
print("all-reduced (averaged):    ", allreduced_grad)
print("identical to full-batch:   ", np.isclose(full_batch_grad, allreduced_grad))

#### 1. What this doesn't fix

DDP scales how much DATA you can train on per step, not how big the MODEL itself can be, every worker still needs to hold a full copy of the model, optimizer state, and gradients in its own memory. Once the model itself is too large for one device's memory, you need model or pipeline parallelism instead.

## Model parallelism vs. pipeline parallelism

#### 0. Core idea

Both split the MODEL (not the data) across devices, for when the model itself doesn't fit on one GPU.

- **Model (tensor) parallelism**: split an individual layer's computation across devices (e.g. each device holds a slice of a huge weight matrix, and they jointly compute one matrix multiply together). Devices must communicate on every single layer, so this needs very fast interconnects (e.g. NVLink within one server) and doesn't scale well across slower network links.
- **Pipeline parallelism**: assign different WHOLE LAYERS (or blocks of layers) to different devices, forming a pipeline, e.g. GPU 0 holds layers 1-8, GPU 1 holds layers 9-16, and so on. A batch is split into smaller micro-batches that flow through the pipeline stage by stage, like an assembly line.

Pipeline parallelism's cost is the "bubble": early in the pipeline, later stages sit idle waiting for the first micro-batch to arrive, and at the end, early stages sit idle waiting for the last micro-batch to finish. The bubble (idle) fraction for `P` pipeline stages and `M` micro-batches is `(P-1) / (M+P-1)`.

In [ ]:
def bubble_fraction(P, M):
    return (P - 1) / (M + P - 1)

for P, M in [(4, 4), (4, 8), (4, 16)]:
    print(f"P={P} stages, M={M} micro-batches -> idle bubble fraction = {round(bubble_fraction(P, M), 4)}")

#### 1. Why more micro-batches, not fewer

With only 4 micro-batches for 4 stages, 43% of every stage's time is wasted bubble (the pipeline hasn't finished filling/draining relative to how little useful work it does). Splitting the same batch into 16 micro-batches instead of 4 shrinks the bubble fraction to 16%, since the fixed fill/drain cost now amortizes over far more useful work. The tradeoff: more micro-batches means each one is smaller, which can hurt hardware utilization within a single micro-batch's own matrix multiplies, so this is tuned, not maximized blindly.

## Sync vs. async gradient aggregation

#### 0. Core idea

**Synchronous** (what DDP's all-reduce above does): every worker's gradient is computed at the CURRENT, shared parameter value, and the update is applied only once every worker has finished and reported in. Simple, correct, but the whole step is only as fast as the SLOWEST worker (a straggler stalls everyone).

**Asynchronous**: workers push gradient updates whenever they finish, without waiting for others, and don't wait for a lock-step round either. No straggler problem, but a worker's gradient was computed against a parameter value that's now STALE, some other worker(s) may have already updated the shared parameters in the meantime. That staleness isn't just noise, it can change how a learning rate that's perfectly stable under synchronous updates behaves once you factor in that delay.

In [ ]:
lr = 0.6  # stable for synchronous gradient descent on this toy loss (see below)

# sync: gradient always computed at the CURRENT parameter value
w_sync = [10.0]
for t in range(10):
    grad = 2 * w_sync[-1]  # d/dw of loss = w^2
    w_sync.append(w_sync[-1] - lr * grad)
print("sync trajectory:            ", [round(v, 3) for v in w_sync])

# async: gradient computed from a value that's already 1 step STALE, applied to the current parameter
w_async = [10.0, 10.0]
for t in range(1, 11):
    stale_w = w_async[t - 1]
    grad = 2 * stale_w
    w_async.append(w_async[t] - lr * grad)
print("async (staleness=1) trajectory:", [round(v, 3) for v in w_async])

#### 1. Reading the result

At `lr=0.6`, synchronous updates converge cleanly to 0 within a handful of steps (oscillating but shrinking). The exact same learning rate under a 1-step-stale asynchronous update DIVERGES, the parameter oscillates with growing amplitude instead of shrinking. Nothing about the loss function or the learning rate changed, only the fact that the gradient was computed against an already-outdated parameter value. This is why async training in practice needs either a smaller learning rate than the synchronous-safe one, a hard cap on how stale a gradient is allowed to get before it's discarded, or simply isn't used at all for large-scale LLM pretraining, where synchronous data parallelism (DDP, ZeRO) is the standard despite the straggler cost.